In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import requests
from IPython.display import Markdown, display
import gradio as gr
import json
import os

In [2]:
load_dotenv(override=True)
openai=OpenAI()

In [ ]:
reader = PdfReader('data/linkedin.pdf')
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
eseoghenabraimah@gmail.com
www.linkedin.com/in/eseoghena-
braimah (LinkedIn)
brymahh.github.io/ (Portfolio)
Top Skills
Data Cleaning
ChatGPT
Education Planning
Certifications
Foundations; Data, Data Everywhere
SQL For Data Science
Supervised Machine Learning:
Regression and Classification
Getting Started with Writing and
Publishing Your Research (2023)
Godsgift Braimah
Data & Everything AI ||Data Scientist || I am Making AI Less
Complicated||Founder @ AI For Kids || If You’re CURIOUS Check
my Featured Section||
Greater Vancouver Metropolitan Area
Summary
Traditionally, this should be a long bio. But frankly, I'm at the point
where I want to create impact.Let my work do the talking!Oh and I
absolutely believe "You too can do great things from a small place."
Experience
The University of British Columbia
11 months
Project Data Analyst
May 2026 - Present (3 months)
Vancouver, BC
►  Data cleaning & preprocessing, including standardization and migration of
on-premises university

In [5]:
with open("data/summary.txt", "r", encoding='utf-8') as f:
    summary = f.read()

In [6]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [7]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - Please tell me about yourself"}
]

In [8]:
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)

display(Markdown(response.choices[0].message.content))

Hello! I'm Godsgift Braimah, a data scientist and AI engineer based in Vancouver, British Columbia. I have a Master's degree in Data Science from the University of British Columbia (UBC), where I specialized in statistical modeling, machine learning, and AI development. 

My expertise lies in turning complex data problems into actionable insights and accessible tools. I have experience in various analytical and predictive frameworks, including time-series forecasting, dimensionality reduction, and natural language processing.

I’ve led notable projects like the Vancouver Neighbourhood Safety Dashboard and co-developed a Python library for analyzing text sentiment shifts. I'm also the founder of AI for Kids, an initiative aimed at helping children, parents, and educators understand AI safely and responsibly.

In addition to my technical skills in programming languages like Python and R, I focus on data cleaning, exploratory analysis, and translating complex findings for non-technical audiences. I'm continuously engaged in learning and community initiatives to stay updated with advancements in AI and machine learning.

If you have any specific questions about my work or skills, feel free to ask!

In [9]:
response.choices[0].message

ChatCompletionMessage(content="Hello! I'm Godsgift Braimah, a data scientist and AI engineer based in Vancouver, British Columbia. I have a Master's degree in Data Science from the University of British Columbia (UBC), where I specialized in statistical modeling, machine learning, and AI development. \n\nMy expertise lies in turning complex data problems into actionable insights and accessible tools. I have experience in various analytical and predictive frameworks, including time-series forecasting, dimensionality reduction, and natural language processing.\n\nI’ve led notable projects like the Vancouver Neighbourhood Safety Dashboard and co-developed a Python library for analyzing text sentiment shifts. I'm also the founder of AI for Kids, an initiative aimed at helping children, parents, and educators understand AI safely and responsibly.\n\nIn addition to my technical skills in programming languages like Python and R, I focus on data cleaning, exploratory analysis, and translating 

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [11]:
chat('Please summarize who you are?', [])

'I am Godsgift Braimah, a data scientist and AI engineer based in Vancouver, British Columbia. With a Master\'s degree in Data Science from the University of British Columbia, I specialize in statistical modeling, machine learning, and AI development. My focus is on transforming complex data problems into actionable insights and accessible tools.\n\nI have significant experience with various programming languages such as Python, R, and SQL, and I have worked on notable projects like the Vancouver Neighbourhood Safety Dashboard and the "You Need a Dictionary" Python package. Additionally, I\'m passionate about continuous learning, actively engaging in the AI community and founding initiatives like AI for Kids, aimed at promoting AI knowledge in children and educators.'

In [12]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
def record_email_tool(email):
    print(f'Tool called to record email: {email}')
    with open("data/emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email recorded successfully"

In [14]:
record_email_tool("eseoghenabraimah@gmail.com")

Tool called to record email: eseoghenabraimah@gmail.com


'Email recorded successfully'

In [15]:
# json to describe the tool

record_email_tool_json = {
    "name" : "record_email_tool",
    "description" : "Record email address provided by a user",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [16]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [17]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(message)
        tool_call = message.tool_calls[0]
        print(tool_call)
        email = json.loads(tool_call.function.arguments).get("email")
        record_email_tool(email)
        messages.append(message)
        print(messages)
        messages.append({"role": "tool", "content": "Email Recorded", "tool_call_id": tool_call.id})
        print(messages)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
 
    return response.choices[0].message.content

In [18]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [19]:
chat("I'd like to get in touch, reach me at eseoghenabraimah@gmail.com", [])

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_eCpitWQp1sOQqnKXNtLKwLN2', function=Function(arguments='{"email":"eseoghenabraimah@gmail.com"}', name='record_email_tool'), type='function')])
ChatCompletionMessageFunctionToolCall(id='call_eCpitWQp1sOQqnKXNtLKwLN2', function=Function(arguments='{"email":"eseoghenabraimah@gmail.com"}', name='record_email_tool'), type='function')
Tool called to record email: eseoghenabraimah@gmail.com
[{'role': 'system', 'content': '\n\n# Your role\n\nYou are a digital twin running on a website, chatting with visitors of the website.\nYou represent the person who\'s website you are on.\nYou answer questions related to their career, background, skills and experience.\n\nHere are the details of the person you are representing:\n\n## **Godsgift Braimah**\n\n**Data Scientist & AI Engineer**\n\n### **Professional Overview**\n\nGodsgift 

'Absolutely — I’ve recorded your email as **eseoghenabraimah@gmail.com**.\n\nThanks for reaching out. If you’d like, you can also share a bit about what you’re interested in discussing — for example:\n- data science / AI opportunities\n- collaboration on a project\n- speaking or mentoring\n- hiring or consulting\n\nI’ll be happy to continue from there.'

In [20]:
# including loop for multiple email calls

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email Recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
 
    return response.choices[0].message.content

In [21]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


### Setting up PushOver Notification ----- For Phone

In [22]:
load_dotenv(override=True)

True

In [23]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [24]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [25]:
push("Today is a great day to be happy!")

Push: Today is a great day to be happy!


In [26]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [27]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [28]:
# json to describe the tool

record_user_details_json = {
    "name" : "record_user_details",
    "description" : "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [29]:
# json to describe the tool

record_unknown_question_json = {
    "name" : "record_unknown_question",
    "description" : "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
            },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [30]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [31]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional info about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],


In [32]:
# function to take a list of tools and run them, when called.

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
            
        results.append({"role": "tool", "content": json.dumps(result),"tool_call_id": tool_call.id })
        
    return results

In [33]:
# updated synthax


def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


In [34]:
globals()

{'__name__': '__main__',
 '__doc__': 'Automatically created module for IPython interactive environment',
 '__package__': None,
 '__loader__': None,
 '__spec__': None,
 '__builtin__': <module 'builtins' (built-in)>,
 '__builtins__': <module 'builtins' (built-in)>,
 '_ih': ['',
  'from dotenv import load_dotenv\nfrom openai import OpenAI\nfrom pypdf import PdfReader\nimport requests\nfrom IPython.display import Markdown, display\nimport gradio as gr\nimport json\nimport os',
  'load_dotenv(override=True)\nopenai=OpenAI()',
  'reader = PdfReader(\'data/linkedin.pdf\')\nlinkedin = ""\nfor page in reader.pages:\n    text = page.extract_text()\n    if text:\n        linkedin += text',
  'print(linkedin)',
  'with open("data/summary.txt", "r", encoding=\'utf-8\') as f:\n    summary = f.read()',
  'system_prompt = f"""\n\n# Your role\n\nYou are a digital twin running on a website, chatting with visitors of the website.\nYou represent the person who\'s website you are on.\nYou answer questions 

In [38]:
display(Markdown(linkedin))

   
Contact
eseoghenabraimah@gmail.com
www.linkedin.com/in/eseoghena-
braimah (LinkedIn)
brymahh.github.io/ (Portfolio)
Top Skills
Data Cleaning
ChatGPT
Education Planning
Certifications
Foundations; Data, Data Everywhere
SQL For Data Science
Supervised Machine Learning:
Regression and Classification
Getting Started with Writing and
Publishing Your Research (2023)
Godsgift Braimah
Data & Everything AI ||Data Scientist || I am Making AI Less
Complicated||Founder @ AI For Kids || If You’re CURIOUS Check
my Featured Section||
Greater Vancouver Metropolitan Area
Summary
Traditionally, this should be a long bio. But frankly, I'm at the point
where I want to create impact.Let my work do the talking!Oh and I
absolutely believe "You too can do great things from a small place."
Experience
The University of British Columbia
11 months
Project Data Analyst
May 2026 - Present (3 months)
Vancouver, BC
►  Data cleaning & preprocessing, including standardization and migration of
on-premises university data for ingestion into analytics platform.
Development and Alumni Engagement Ambassador
September 2025 - April 2026 (8 months)
Vancouver, BC
►  Support UBC Development & Alumni Engagement initiatives by assisting
with alumni outreach, donor relations, and engagement events. 
► Work collaboratively with staff and ambassadors as a professional
representative of the University of British Columbia, using interpersonal and
communication skills to support positive alumni experiences.
AI For Kids
Founder
October 2024 - June 2026 (1 year 9 months)
►  On a mission to ensure children, parents, educators and teachers
understand the foundations of Artificial Intelligence, the risks and potentials
involved. And through this, leverage AI safely and responsibly for promoting
creativity, growth and education.
  Page 1 of 6   
► Leading cross-functional collaborations with schools, partner organizations
and NGOs to inspire the next generation of African children to be creators and
not just consumers.
AMDARI
Head of Internships & Data Professional
December 2024 - April 2025 (5 months)
Alberta, Canada
► Pioneered data-driven quality control processes to evaluate technical intern
projects, identifying areas for continuous improvement in training delivery and
program outcomes.
► Directed the daily operational activities for the Remote Apprenticeship
Virtual Experience (RAVE) team, leading over 150 interns and aligning project
delivery with high-level company goals.
► Supervised specialists across multiple technical domains, ensuring the
delivery of high-quality training and real-world guided solutions.
Xterns by Darey.io
Data Analytics and AI Engineer
October 2024 - November 2024 (2 months)
Lagos State, Nigeria
► Collaborated in engineering AI algorithms for skill tracking and personalized
learning paths, including a semantic comparison system for user assessment.
► Researched AI advancements to improve platform capabilities for over 100
learners.
DSNai - Data Science Nigeria
Data Scientist (AI Research & Innovation)
March 2024 - October 2024 (8 months)
Lagos State, Nigeria
► Contributed to the development of social-impact AI solutions, including a
Retrieval-Augmented Generation (RAG) system utilizing LLMs and LangChain
for streamlined query answering.
► Collaborated on the deployment of a data-driven chatbot solution providing
vital reproductive and maternal information for communities in northern
Nigeria.
  Page 2 of 6   
Nebiant Analytics
Data Scientist
June 2024 - August 2024 (3 months)
Ikeja, Lagos State, Nigeria
► Created comprehensive curriculum and video resources to reduce entry
barriers for Africans transitioning into the Data Science.
► Developed advanced instructional resources for Excel and data analysis to
enhance learners' technical proficiency and analytical skills
10Alytics
1 year 6 months
Senior Data Scientist
April 2023 - February 2024 (11 months)
Manchester Area, United Kingdom
► Directed the full-stack data science division, managing intelligent reporting
and the provision of data-driven solutions for global partners and clients.
► Facilitated data science masterclasses for over 1,000 individuals across
Africa, Europe, and North America, focusing on data preprocessing,
visualization, exploratory data analysis, machine learning and insight
generation.
► Coordinated monthly technical cohorts, hosting review sessions on data
processing using Python, SQL, Tableau, and Excel.
Data Scientist
September 2022 - March 2023 (7 months)
Manchester
Coordinated the monthly data science cohorts, managing the engagement of
over 1000 enthusiasts across Africa and Europe.
EdoInnovates
5 months
Data Science Intern
September 2022 - December 2022 (4 months)
Benin City, Edo, Nigeria
Trained over 70 persons on Data Analysis and the Structured Query
Language(SQL), educating them on the writing of queries and techniques in
querying your data.
  Page 3 of 6   
Intern
September 2022 - September 2022 (1 month)
Benin City, Edo, Nigeria
Trained over 200 adults on the basics of spreadsheets applications; Microsoft
Excel and Google sheets. Provided an understanding of the tools, functions
and methods utilized in analyzing and visualization of tabular data.
Tutor
August 2022 - August 2022 (1 month)
Benin City, Edo, Nigeria
InoKiTech Kids Bootcamp:
Tutored over 91 children within the ages of 6-11 on the basics and introductory
concepts of computer programming using Google’s computer science
curriculum for Scratch. They had an understanding of the processes involved
in writing codes and fixing bugs.
EdoInnovates
Data Science Entry Level Intern
March 2022 - June 2022 (4 months)
Benin City, Edo, Nigeria
• Acquisition of practical knowledge on data science, procedures of data
cleaning, exploration, and visualization with python. Creation of dummy
models for tackling regression and classification tasks. Understanding of
Entity Relationship Diagrams and querying of data with the Structured Query
Language (SQL).  Data visualization and analysis with PowerBI and Microsoft
Excel.
• Working in a team of eight (8) persons in analyzing data obtained from
Nigerian graduates, determination of a business problem from the data
and proffering data driven solutions and presentation of findings to the
administrators at Edo Innovation hub.
The Build Links Limited
Intern
April 2021 - July 2021 (4 months)
Lagos, Nigeria
Had the privilege of having my undergraduate internship with The Build Links
Limited at Victoria Island, Lagos. I obtained an understanding of the pile drilling
process for deep foundations. 
Role:
  Page 4 of 6   
• Organization of the site office and documents, recording changes and
progress of construction procedures daily. Ensuring copies of the drawings
were available for construction updates.
• Supervision of the pile drilling process to ensure that the mast and the drilling
pipes were vertically aligned, and the drilling bit entered directly above the peg.
• Documentation of the Pile Integrity Test (PIT) to keep track of the piles
tested.
• Recording of tools and construction materials used daily.
University of Benin
Student Vice President
March 2018 - June 2019 (1 year 4 months)
Edo State, Nigeria
Vice President of the "Association of Civil and Structural Engineering
Students", University of Benin Chapter; with the responsibility of coordinating
the activities of the student body and serving as a middle man between the
institution and the student body.
Yisab Nigeria Limited
Student Intern
February 2019 - March 2019 (2 months)
Edo State, Nigeria
Due to the restructure of Federal Polytechnic Auchi, various faculties and
departmental lecture halls' constructions were launched. I acquired basic
knowledge on setting out, landscaping, block work,  foundations, form works,
steel truss roofing system and the various finishes employed in concrete
structures. 
University of Benin
Student Intern
September 2017 - October 2017 (2 months)
Edo State, Nigeria
Every year, the University of Benin facilitates an industrial training for first year
engineering students transitioning to their second year with the sole aim of
welcoming students into the Faculty of Engineering. This experience gave a
clear insight and introduction of my course work.
Education
The University of British Columbia
  Page 5 of 6   
Master of Data Science - MDS  · (September 2025)
University of Benin
Bachelor of Engineering - BE  · (2016 - December 2022)
  Page 6 of 6

## Final

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [40]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [41]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: record_unknown_question
Push: Recording Tell me about your love for dancing asked that I couldn't answer


/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/s

Tool called: record_user_details
Push: Recording interest from Freda with email freda@telus.ai and notes User wants to discuss an opportunity and would like to be contacted.


/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: record_user_details
Push: Recording interest from shaorn with email sharon@mars.com and notes User said they may name is shaorn and provided email sharon@mars.com. Spelling may be 'sharon' or 'shaorn'.


/Users/braimah/Documents/MDS_UBS/projects/your_digital_twin_agent/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
